# Flood Stats
- Data: Gauge, Discharge, Rainfall
- Note: Only STID RA101

## Goals
10/31 Meeting
- Clean data
- Get daily max values (creek values)
- Separate into 3 levels:
    - Minor flood stage: 8.5 ft
    - Moderate flood stage: 9.5 ft
    - Major flood stage: 11 ft
    - Minimum operating limit: 0.5 ft

## Data Dictionary
- date: date of measurement
- hour: hour of measurement
- stid: regional location
- gauge: water level in feet
- discharge: discharge level in cubic feet per second
- rainfall: rainfall level in inches
- rainfall_ft: rainfall level in feet

### Data Sources
- [Gauge and Discharge](https://waterdata.usgs.gov/monitoring-location/01464000/#parameterCode=00060&startDT=2005-10-01&endDT=2022-10-18)
- [Rainfall](https://njdep.rutgers.edu/rainfall/)

### Setup

In [ ]:
import pandas as pd
from pandas import DataFrame
import numpy as np
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
# Display entire dataframe
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# cd '/Users/shifraisaacs/Documents/Externship/cgi_flood_prediction_mitigation'

### Load Data

In [ ]:
combined = pd.read_csv('combined_rainfall_discharge_gauge.csv', index_col=0)
print(combined.shape)
combined.head()

### Data Cleaning and Manipulation

In [ ]:
# Rearrange columns and fix spelling mistake
combined.columns = combined.columns.str.lower()
combined = combined[['date', 'stid', 'guage', 'discharge', 'rainfall']]
combined = combined.rename(columns={'guage': 'gauge', 'rainfall': 'rainfall_inches'})
combined.head()

In [ ]:
combined['rainfall_ft'] = round(combined['rainfall_inches']/12, 2)

In [ ]:
combined['date'] = pd.to_datetime(combined['date'], infer_datetime_format=True)

In [ ]:
combined.info()

In [ ]:
# Confirm no null values
combined.isnull().sum()

### Descriptive Stats

In [ ]:
combined.describe()

### Correlational Analysis
Strong correlation between gauge and discharge; minimal correlation with rainfall

In [ ]:
combined.corr()

### Group data to get max levels for each day

In [ ]:
daily_max = combined.groupby(by='date').max().reset_index()
daily_max.head()

In [ ]:
daily_max.describe()

In [ ]:
daily_max.to_csv('Data/Daily_Max_Vals.csv')

## Create data subsets

In [ ]:
# Minimum operating limit
op_limit = daily_max[daily_max['gauge'] >= 0.5]
print(op_limit.shape)
op_limit.head()

In [ ]:
# Minor flood stage
minor_filter = np.where((daily_max['gauge'] >= 8.5) & (daily_max['gauge'] < 9.5))
minor_flood = daily_max.loc[minor_filter]
print(minor_flood.shape)
minor_flood.head()

In [ ]:
# Moderate flood stage
moderate_filter = np.where((daily_max['gauge'] >= 9.5) & (daily_max['gauge'] < 11))
moderate_flood = daily_max.loc[moderate_filter]
print(moderate_flood.shape)
moderate_flood.head()

In [ ]:
# Major flood stage
major_filter = np.where(daily_max['gauge'] > 11)
major_flood = daily_max.loc[major_filter]
print(major_flood.shape)
major_flood.head()